# EDA SINAN — Brasil

Dados de anos encerrados, agregados por residência. Execute primeiro `python -m utils.deng_download` e `python -m utils.analysis_pipeline`. As saídas anteriores foram invalidadas após correção da classificação histórica. `DENGUE` não é diagnóstico final: usamos `CON_CLASSI` (até 2006) e `CLASSI_FIN` (demais anos).

**Período anual:** `ano` nos contratos SINAN identifica o ano-fonte (`periodo_referencia = ano_fonte_sinan`), não necessariamente um ano civil completo. Datas em anos adjacentes são preservadas e auditadas. A população usa o ano de referência; as séries diárias climáticas permanecem separadas, pela data efetiva de notificação.

A coorte original inclui uma notificação de 2008 com data corrompida. A execução usa `allow_undated=True`: o registro permanece no período-fonte anual e na auditoria, sem inventar dia ou mês; gráficos por data não o incluem.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "utils/analysis_pipeline.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import polars as pl
from IPython.display import display

import plotly.express as px
from utils.analysis_pipeline import PROCESSED_DIR, build_sinan_outputs, scan_sinan
outputs = build_sinan_outputs(allow_undated=True)
panel = pl.read_parquet(PROCESSED_DIR / "sinan_populacao_municipio_ano.parquet")
raw = scan_sinan()

/Users/thitav/Documents/Pysus-IBGE/utils/analysis_pipeline.py:410: UserWarning: 1 registro(s) sem data válida mantido(s) apenas pelo ano-fonte; a exportação mensal conserva ano/mês civil desconhecido. Consulte auditoria_sinan_datas_invalidas.csv.
  warnings.warn(


## Qualidade e universo

Notificações não são necessariamente casos confirmados. Sem fonte anual não se infere zero. Municípios sem notificação recebem zero somente em anos com fonte presente. Registros sem residência válida ficam na auditoria; não são distribuídos entre municípios.

In [2]:
display(outputs["qualidade"])
annual = panel.group_by("ano").agg(pl.col("notificacoes").sum(), pl.col("casos_confirmados").sum()).sort("ano")
display(annual)
px.line(annual.to_pandas(), x="ano", y=["notificacoes", "casos_confirmados"], markers=True).show()

arquivo,ano_arquivo,registros,codigo_municipio_residencia_invalido,codigo_municipio_notificacao_invalido,codigo_municipio_invalido,municipio_residencia_notificacao_divergente,data_invalida,ano_informado_divergente,ano_data_divergente
str,i32,u32,u32,u32,u32,u32,u32,u32,u32
"""DENGBR00.parquet""",2000,172855,1,0,1,7207,0,48,0
"""DENGBR01.parquet""",2001,488590,34,0,34,14012,0,243,0
"""DENGBR02.parquet""",2002,897093,255,0,255,42574,0,202,0
"""DENGBR03.parquet""",2003,416609,311,0,311,16742,0,20,0
"""DENGBR04.parquet""",2004,136867,67,0,67,5507,0,0,0
…,…,…,…,…,…,…,…,…,…
"""DENGBR19.parquet""",2019,2261956,52,0,52,138127,0,1809,1809
"""DENGBR20.parquet""",2020,1495117,35,0,35,69046,0,6800,6800
"""DENGBR21.parquet""",2021,1024635,77,0,77,55955,0,1466,1466


ano,notificacoes,casos_confirmados
i32,u64,u64
2000,172854,99349
2001,488556,286610
2002,896838,481260
2003,416298,227424
2004,136800,63483
…,…,…
2019,2261904,1311737
2020,1495082,793006
2021,1024558,455711


## Sazonalidade

Grade completa de dias dos anos-fonte, incluindo dias sem notificação registrada. A média mensal abaixo é por dia de calendário; a soma mensal é publicada separadamente.

In [3]:
undated_count = raw.filter(pl.col("data_notificacao").is_null() & pl.col("codigo_municipio").is_not_null()).select(pl.len()).collect().item()
print("Datas desconhecidas: mantidas no ano-fonte e excluídas apenas do gráfico diário:", undated_count)
from datetime import date
years = outputs["qualidade"]["ano_arquivo"].to_list()
calendar = pl.concat([pl.DataFrame({"Dia": pl.date_range(date(y, 1, 1), date(y, 12, 31), eager=True)}) for y in years])
daily = raw.filter(pl.col("codigo_municipio").is_not_null()).group_by("data_notificacao").agg(pl.len().alias("notificacoes")).rename({"data_notificacao":"Dia"}).collect()
daily = calendar.join(daily, on="Dia", how="left", validate="1:1").with_columns(pl.col("notificacoes").fill_null(0)).sort("Dia")
monthly = daily.with_columns(pl.col("Dia").dt.year().alias("ano"), pl.col("Dia").dt.month().alias("mes")).group_by("ano", "mes").agg(pl.col("notificacoes").sum().alias("notificacoes_mes"), pl.col("notificacoes").mean().alias("media_por_dia_calendario")).sort("ano", "mes")
display(monthly)
px.line(monthly.to_pandas(), x="mes", y="media_por_dia_calendario", color="ano").show()

Datas desconhecidas: mantidas no ano-fonte e excluídas apenas do gráfico diário: 1


ano,mes,notificacoes_mes,media_por_dia_calendario
i32,i8,u32,f64
2000,1,14207,458.290323
2000,2,24740,853.103448
2000,3,32163,1037.516129
2000,4,28897,963.233333
2000,5,24669,795.774194
…,…,…,…
2023,8,33020,1065.16129
2023,9,24391,813.033333
2023,10,28138,907.677419


## Classificação, raça/cor, sexo e evolução das notificações

In [4]:
for column in ["status_classificacao", "raca", "sexo", "evolucao"]:
    summary = raw.group_by(column).len(name="registros").sort("registros", descending=True).collect()
    display(summary)
    px.bar(summary.to_pandas(), x=column, y="registros", title=f"Notificações por {column}").show()

status_classificacao,registros
str,u32
"""confirmado""",14280945
"""descartado""",7216926
"""inconclusivo""",2913626
"""nao_preenchido""",464940
"""outro_codigo""",154564


raca,registros
str,u32
"""branca""",8109925
"""parda""",7629891
"""ignorado""",5062069
"""nao_preenchido""",3001801
"""preta""",958584
"""amarela""",206164
"""indigena""",62567


sexo,registros
str,u32
"""F""",13747542
"""M""",11254013
"""I""",28664
null,781
"""O""",1


evolucao,registros
str,u32
"""cura""",18551731
"""nao_preenchido""",5363513
"""ignorado""",776927
"""nao_se_aplica""",306096
"""obito_outras_causas""",17613
"""obito_pelo_agravo""",13360
"""obito_em_investigacao""",1761


## Taxas municipais anuais

População residente do mesmo ano e chave IBGE. Nunca se divide a soma de vários anos por uma única população censitária. A seleção abaixo usa 2010; altere apenas para um ano com denominador disponível.

In [5]:
YEAR = 2010
municipal = panel.filter((pl.col("ano") == YEAR) & pl.col("correspondencia_populacao"))
if not municipal.height:
    raise ValueError(f"Sem denominador válido para {YEAR}")
display(municipal.select("codigo_municipio", "municipio", "populacao", "notificacoes", "casos_confirmados", "taxa_confirmados_100k").sort("taxa_confirmados_100k", descending=True))
px.bar(municipal.sort("taxa_confirmados_100k", descending=True).head(10).to_pandas(), x="taxa_confirmados_100k", y="municipio", orientation="h", title=f"Taxa de confirmados por 100 mil residentes — {YEAR}").show()
states = municipal.with_columns(pl.col("codigo_municipio").str.slice(0, 2).alias("UF_codigo")).group_by("UF_codigo").agg(pl.col("notificacoes").sum(), pl.col("casos_confirmados").sum(), pl.col("populacao").sum()).with_columns((pl.col("casos_confirmados") / pl.col("populacao") * 100_000).alias("taxa_confirmados_100k"))
display(states)

codigo_municipio,municipio,populacao,notificacoes,casos_confirmados,taxa_confirmados_100k
str,str,u64,u64,u64,f64
"""120040""","""Rio Branco - AC""",336038,37100,32336,9622.72124
"""314050""","""Martinho Campos - MG""",12611,1252,1152,9134.882246
"""510719""","""Ribeirãozinho - MT""",2199,265,200,9095.043201
"""313880""","""Luz - MG""",17486,1762,1546,8841.358801
"""520320""","""Barro Alto - GO""",8716,795,766,8788.435062
…,…,…,…,…,…
"""521350""","""Monte Alegre de Goiás - GO""",7730,91,0,0.0
"""521385""","""Morro Agudo de Goiás - GO""",2356,0,0,0.0
"""521490""","""Nova Roma - GO""",3471,2,0,0.0


UF_codigo,notificacoes,casos_confirmados,populacao,taxa_confirmados_100k
str,u64,u64,u64,f64
"""14""",11785,6990,450479,1551.681654
"""27""",54488,37877,3120494,1213.814223
"""28""",1978,545,2068017,26.353749
"""42""",644,166,6248436,2.656665
"""16""",4462,2378,669526,355.176647
…,…,…,…,…
"""31""",266855,190659,19597330,972.88253
"""51""",45740,35937,3035122,1184.038072
"""33""",42419,21181,15989929,132.464628


## Conclusões

A série corrigida usa classificação final em todas as eras. As contagens antigas baseadas no antecedente `DENGUE` não fundamentam conclusões sobre magnitude ou tendência em 2000–2006. O período anual é o ano-fonte, e a localização é a residência. As taxas são descritivas: refletem vigilância, diagnóstico e transmissão, sem demonstrar causalidade ou eliminar subnotificação.